# 03｜CAPM 的現實：平坦的 SML 與低波動溢酬

**學術來源：**
- Black, Jensen & Scholes (1972)：SML 比 CAPM 預測更平坦
- Frazzini & Pedersen (2014), *Betting Against Beta*, Journal of Financial Economics

CAPM 預測：高 β 股票應有更高報酬（承擔更多市場風險）。

**現實是**：高 β 股票的報酬並不像 CAPM 預測那樣高；低 β 股票的報酬卻超出預期。

這個現象稱為 **「低波動異象（Low-Volatility Anomaly）」**，
也催生了一個可交易策略：**Betting Against Beta（BAB）**。

## 「高風險才有高報酬」——你同意嗎？

這句話聽起來天經地義，對吧？

存定期比活期利率高，因為你犧牲了流動性。
買高收益債比投資等級債利率高，因為違約風險更大。
以此類推：買波動更大的股票，應該得到更高的報酬補償。

**這個邏輯就是 CAPM 的核心。** 1964 年 Sharpe 把它公式化（他後來得了諾貝爾獎）：
股票預期報酬 = 無風險利率 + β × 市場溢酬。β 越大（越劇烈波動）→ 報酬應該越高。

直觀、優雅、邏輯自洽。

但實際數據呢？

我們把幾十年的美股資料按 β 值分組，看看高 β 組是不是真的報酬更高……
結果有點令人尷尬。

## 🎯 學習目標
1. 理解 CAPM 的核心預測：風險（β）應對應更高的預期報酬
2. 用 FF25 投資組合數據驗證「安全市場線（SML）」實際比理論平坦
3. 認識 Betting Against Beta（BAB）因子的邏輯與實證績效

## 1｜CAPM 複習：預測與現實的落差

**CAPM 公式：**
$$E[R_i] - R_f = \beta_i \cdot (E[R_m] - R_f)$$

所以在「安全市場線（SML）」上：
- β=0：報酬 = 無風險利率
- β=1：報酬 = 市場報酬
- β=2：報酬 = 市場溢酬的兩倍

**實證發現：** SML 斜率遠低於 CAPM 預測，大約是預測值的 40–60%。
換言之，β 每增加 1，實際額外報酬遠小於理論值。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = [
    'Microsoft YaHei', 'SimHei', 'Heiti TC',
    'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif'
]
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
from scipy import stats
import warnings, pathlib
warnings.filterwarnings('ignore')

DATA_DIR = pathlib.Path('data')
DATA_DIR.mkdir(exist_ok=True)

FF25_CSV = DATA_DIR / 'ff25_portfolios.csv'
FF3_CSV  = DATA_DIR / 'ff3_factors.csv'

# 載入 FF3
if FF3_CSV.exists():
    ff3 = pd.read_csv(FF3_CSV, index_col=0, parse_dates=True)
    print(f'FF3 從快取載入：{len(ff3)} 筆')
else:
    import pandas_datareader.data as web
    raw3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1963-07')[0]
    ff3 = raw3 / 100
    ff3.index = pd.to_datetime(ff3.index.to_timestamp())
    ff3.to_csv(FF3_CSV)
    print(f'FF3 下載完成：{len(ff3)} 筆')

# 載入 FF25 (25 Portfolios sorted by Size x BM)
if FF25_CSV.exists():
    ff25 = pd.read_csv(FF25_CSV, index_col=0, parse_dates=True)
    print(f'FF25 從快取載入：{ff25.shape}')
else:
    import pandas_datareader.data as web
    raw25 = web.DataReader('25_Portfolios_5x5', 'famafrench', start='1963-07')[0]
    ff25 = raw25 / 100
    ff25.index = pd.to_datetime(ff25.index.to_timestamp())
    ff25.to_csv(FF25_CSV)
    print(f'FF25 下載完成：{ff25.shape}')

# 對齊時間
common = ff3.index.intersection(ff25.index)
ff3  = ff3.loc[common]
ff25 = ff25.loc[common]
print(f'\n對齊後：{len(common)} 個月（{common[0].year}–{common[-1].year}）')

## 2｜計算 25 個投資組合的 β 與平均超額報酬

In [ ]:
mkt_rf = ff3['Mkt-RF'].values
rf     = ff3['RF'].values

results = []
for col in ff25.columns:
    port_ret = ff25[col].values
    excess   = port_ret - rf
    slope, intercept, r, p, se = stats.linregress(mkt_rf, excess)
    ann_excess = excess.mean() * 12
    results.append({
        'portfolio': col,
        'beta':      slope,
        'alpha':     intercept * 12,
        'ann_excess': ann_excess,
        'r2':        r**2
    })

res_df = pd.DataFrame(results)
print(res_df.sort_values('beta')[['portfolio','beta','ann_excess','alpha','r2']].to_string(index=False))

## 3｜安全市場線：預測 vs 現實

In [ ]:
mkt_premium = ff3['Mkt-RF'].mean() * 12  # CAPM 預測斜率
rf_ann      = ff3['RF'].mean() * 12

beta_range = np.linspace(res_df['beta'].min() - 0.1, res_df['beta'].max() + 0.1, 100)
capm_pred  = mkt_premium * beta_range

# 實際 SML（OLS 擬合）
act_slope, act_intercept, *_ = stats.linregress(res_df['beta'], res_df['ann_excess'])
actual_line = act_intercept + act_slope * beta_range

fig, ax = plt.subplots(figsize=(10, 7))

ax.plot(beta_range, capm_pred,  '--', color='#F44336', linewidth=2,
        label=f'CAPM 預測 SML（斜率={mkt_premium*100:.1f}%）')
ax.plot(beta_range, actual_line, '-', color='#2196F3', linewidth=2,
        label=f'實際 SML（斜率={act_slope*100:.1f}%）')
ax.scatter(res_df['beta'], res_df['ann_excess'],
           c='#4CAF50', s=60, zorder=5, label='FF25 投資組合')
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')

coverage = act_slope / mkt_premium
ax.set_title(f'安全市場線：預測 vs 現實\n（實際斜率 ≈ CAPM 預測的 {coverage:.0%}）',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Beta（系統性風險）', fontsize=12)
ax.set_ylabel('年化超額報酬 (%)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# 格式化 y 軸為百分比
import matplotlib.ticker as mticker
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x*100:.1f}%'))

plt.tight_layout()
plt.savefig('data/sml_reality.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'CAPM 預測市場溢酬：{mkt_premium*100:.2f}%/年')
print(f'實際 SML 斜率：    {act_slope*100:.2f}%/年')
print(f'實際斜率 = CAPM 預測的 {coverage:.1%}')

## 4｜BAB 代理策略：低 β 組合 vs 高 β 組合

**Frazzini & Pedersen（2014）** 的 BAB 因子：
- 買入低 β 股票（去槓桿化到 β=1）
- 放空高 β 股票（去槓桿化到 β=1）

我們用 FF25 的最低 β 五個組合 vs 最高 β 五個組合做簡單代理。

In [ ]:
# 按 beta 排序，取最低/最高五個
res_sorted = res_df.sort_values('beta')
low_beta_ports  = res_sorted.head(5)['portfolio'].tolist()
high_beta_ports = res_sorted.tail(5)['portfolio'].tolist()

low_ret  = ff25[low_beta_ports].mean(axis=1)
high_ret = ff25[high_beta_ports].mean(axis=1)
bab_proxy = low_ret - high_ret

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 累積報酬
for name, s, color in [('低 β 組合', low_ret, '#4CAF50'),
                        ('高 β 組合', high_ret, '#F44336'),
                        ('市場', ff3['Mkt-RF']+ff3['RF'], '#9E9E9E')]:
    cum = (1 + s).cumprod()
    axes[0].plot(cum.index, cum, label=name, color=color,
                 linewidth=2 if name != '市場' else 1.2,
                 alpha=1.0 if name != '市場' else 0.6)

axes[0].set_yscale('log')
axes[0].set_title('低 β vs 高 β 組合累積報酬', fontsize=13, fontweight='bold')
axes[0].set_ylabel('累積倍數（對數）')
axes[0].legend()
axes[0].grid(alpha=0.3)

# BAB 代理因子
cum_bab = (1 + bab_proxy).cumprod()
axes[1].plot(cum_bab.index, cum_bab, color='#9C27B0', linewidth=1.5)
axes[1].fill_between(cum_bab.index, 1, cum_bab,
                     where=(cum_bab >= 1), alpha=0.2, color='#4CAF50')
axes[1].fill_between(cum_bab.index, 1, cum_bab,
                     where=(cum_bab < 1),  alpha=0.2, color='#F44336')
axes[1].axhline(1, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('BAB 代理因子（低β − 高β）累積報酬', fontsize=13, fontweight='bold')
axes[1].set_ylabel('累積倍數')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('data/bab_proxy.png', dpi=150, bbox_inches='tight')
plt.show()

for name, s in [('低β組合', low_ret), ('高β組合', high_ret), ('BAB代理', bab_proxy)]:
    ann_r = s.mean() * 12
    ann_v = s.std() * np.sqrt(12)
    print(f'{name}: 年化報酬={ann_r*100:.2f}%，夏普={ann_r/ann_v:.3f}')

## 5｜這跟你有什麼關係？

### 三個反直覺但重要的結論

**① 「高風險 = 高報酬」在股票 β 上並不成立**

CAPM 是財務學的基石，但它的核心預測——高 β 股票帶來高報酬——被 60 年數據反駁。
實際上，SML 的斜率約是 CAPM 預測的 40–60%。

**② 為什麼低 β 股反而有好報酬？**

Frazzini & Pedersen 的解釋：
- 有槓桿限制的機構投資人（退休基金等）**只能買股票，不能用槓桿**
- 他們選擇高 β 股票來「放大」曝險 → 把高 β 股推得過貴
- 低 β 股被忽視 → 相對低估

**③ 實務應用：低波動 ETF 的學術根基**

| 工具 | 說明 |
|------|------|
| **USMV**（iShares MSCI Min Vol USA） | 最大化分散同時最小化波動 |
| **SPLV**（Invesco S&P 500 Low Vol） | 直接選 S&P500 中最低波動的 100 檔 |
| **個股思維** | 選 β < 0.8 的優質股，而非追逐高 β 飆股 |

注意：低波動策略在牛市中通常落後大盤，但長期夏普比率更高——你願意接受嗎？

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| SML 斜率 | 實際約為 CAPM 預測的 40–60% |
| BAB 異象 | 低 β 投資組合長期夏普比率優於高 β |
| 機制解釋 | 槓桿限制→機構追逐高β→高β股被推高估值 |
| 個人應用 | 低波動 ETF（USMV、SPLV）有學術依據 |

> **下一章：** Trinity Study——你的退休金能撐多久？